In [14]:
import numpy as np
import scipy as sp
from ngsolve import *

In [7]:
def fomega(x):
    ell = 0.3
    return np.tan(x) - (ell * x) / (ell**2 * x**2 - 1)

# find one root
sol = sp.optimize.root_scalar(
    fomega,
    bracket = (1.6, 3.0),
    method = "brentq"
)

print(sol.root)

2.24966907917715


In [10]:

def fomega(w, ell):
    return np.tan(w) - (ell * w) / (ell**2 * w**2 - 1)


def find_all_roots(ell, w_min, w_max, points_per_interval=200):
    if ell == 0:
        raise ValueError("ell must be nonzero.")

    # tan(w) is singular at pi/2 + k*pi
    k_min = int(np.floor((w_min - np.pi / 2) / np.pi))
    k_max = int(np.ceil((w_max - np.pi / 2) / np.pi))

    tan_poles = [
        np.pi / 2 + k * np.pi
        for k in range(k_min, k_max + 1)
        if w_min < np.pi / 2 + k * np.pi < w_max
    ]

    # Rational term is singular at w = +/- 1/ell
    rational_poles = [
        pole
        for pole in (-1 / ell, 1 / ell)
        if w_min < pole < w_max
    ]

    # These points divide the domain into continuous intervals
    boundaries = sorted(
        {w_min, w_max, *tan_poles, *rational_poles}
    )

    roots = []

    for left, right in zip(boundaries[:-1], boundaries[1:]):
        # Stay a small distance away from singular endpoints
        eps = 1e-10 * max(1.0, abs(left), abs(right))

        a = left + eps
        b = right - eps

        if a >= b:
            continue

        # Sample inside this continuous interval
        grid = np.linspace(a, b, points_per_interval)
        values = fomega(grid, ell)

        for x1, x2, y1, y2 in zip(
            grid[:-1], grid[1:], values[:-1], values[1:]
        ):
            if not (np.isfinite(y1) and np.isfinite(y2)):
                continue

            # Capture a root that lands almost exactly on a grid point
            if np.isclose(y1, 0.0, atol=1e-12):
                roots.append(x1)

            # Brent's method requires a sign-changing bracket
            if y1 * y2 < 0:
                result = sp.optimize.root_scalar(
                    fomega,
                    args=(ell,),
                    bracket=(x1, x2),
                    method="brentq",
                )

                if result.converged:
                    roots.append(result.root)

    # Check the final sampled endpoint
    if np.isclose(fomega(w_max, ell), 0.0, atol=1e-12):
        roots.append(w_max)

    # Sort and remove numerical duplicates
    roots = sorted(roots)

    unique_roots = []
    for root in roots:
        if not unique_roots or not np.isclose(
            root, unique_roots[-1], atol=1e-9, rtol=1e-9
        ):
            unique_roots.append(root)

    return np.array(unique_roots)

In [15]:
ell = 0.3

roots = find_all_roots(
    ell=ell,
    w_min=-20,
    w_max=20,
)
omegas = roots
print(roots)

[-19.02834916 -15.92349078 -12.83791144  -9.79230172  -6.85071699
  -4.25447479  -2.24966908   0.           2.24966908   4.25447479
   6.85071699   9.79230172  12.83791144  15.92349078  19.02834916]


In [ ]:
def gen_eig(ell, omega, x):
    eigval = (2 * ell) / (ell**2 * omega**2 + 1)
    protovec = np.sin(omega * x) + ell * omega * np.cos(omega * x)
    